# Get preds on every hyper-param combination across folds/repeats

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.utils import ALPHA_LIST, L1_LIST
from src.swarm import run_swarms, make_full_lr, make_full_shot

In [ ]:
SEG_TYPES = [
    "turns",
    "pt_noshort",
]
MODEL_NAMES = [
    "Llama-3.3-8B-Instruct",
    "Llama-3.3-70B-Instruct",
]
IMPACT_TYPES = [
    "self",
    "parent",
]
PROB_AGG_TYPES = [
    "block",
    "max",
    "avg",
    "proba-limited",
    "binary",
]
DESC_TYPES = [
    "original",
    "augment",
    "augment_neg",
]
LINEAR_TYPES = [
    # "elastic",
    "ridge",
]
SAVE_DIR = BASE_PATH / "perm_res" / "preds"

START_PERM = 0
END_PERM = 10  # non-inclusive
INCLUDE_BASELINE = True

## Get model preds at every:

- (Self, Parent) x (block, avg, max, binary, proba-limited) x (8B, 70B) x ((turns, pt_noshort) / (original, negative, augment)) x Alpha Val x Repeat (1-10) x test fold (1-3) combination for:
    - 2 Layers
    - 3 Layers

In [ ]:
## Note that split_tol, n_splits, n_repeats set as globals in swarm.py
N_JOBS = 1
N_THREADS = 2
TUNE_FLAG = "False"
SWARM_DIR = BASE_PATH / "swarm_dir" / "preds"
metric_agg = "combined_folds"  # combined_folds or per_fold

In [ ]:
full_cmd_list = []
cmd_dir = SWARM_DIR / "commands"
for impact_type in IMPACT_TYPES:
    swarm_path = cmd_dir / f"{impact_type}.swarm"
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    if swarm_path.exists():
        swarm_path.unlink()
    cmd_list = []
    for perm_dir in SAVE_DIR.iterdir():
        perm_name = perm_dir.name
        if perm_name != "baseline":
            # Set starting and end perm numbers to control where we are permuting
            perm_num = int(perm_name.split("_")[1].strip(""))
            if (perm_num < START_PERM) or (perm_num >= END_PERM):
                continue
            shuffle = "text"
        else:
            if not INCLUDE_BASELINE:
                continue
            shuffle = "none"
        for agg_type in PROB_AGG_TYPES:
            for linear_type in LINEAR_TYPES:
                if linear_type == "ridge":
                    l1_list = None
                else:
                    l1_list = L1_LIST
                for n_layers in [2, 3]:
                    base_save_dir = perm_dir / f"{n_layers}_layers" / linear_type
                    ## Full model (block, max, avg)
                    for model_name in MODEL_NAMES:
                        if agg_type in ["block", "max", "avg"]:
                            lr_cmd_list = make_full_lr(
                                seg_types=SEG_TYPES,
                                model_name=model_name,
                                impact_type=impact_type,
                                agg_type=agg_type,
                                base_save_dir=base_save_dir,
                                perm_dir=perm_dir,
                                n_jobs=N_JOBS,
                                alpha_list=ALPHA_LIST,
                                l1_list=l1_list,
                                n_threads=N_THREADS,
                                tune_flag_str=TUNE_FLAG,
                                n_layers=n_layers,
                                linear_type=linear_type,
                                metric_agg=metric_agg,
                                perm_shuffle=shuffle,
                            )
                            cmd_list += lr_cmd_list
                        ## 0 shot
                        else:
                            for desc_type in DESC_TYPES:
                                cmd_str = make_full_shot(
                                    model_name=model_name,
                                    impact_type=impact_type,
                                    temp=0.6,
                                    pred_type=agg_type,  # one of binary, proba-limited
                                    desc_type=desc_type,
                                    perm_dir=perm_dir,
                                    base_save_dir=base_save_dir,
                                    n_jobs=N_JOBS,
                                    alpha_list=ALPHA_LIST,
                                    l1_list=l1_list,
                                    n_threads=N_THREADS,
                                    tune_flag_str=TUNE_FLAG,
                                    n_layers=n_layers,
                                    linear_type=linear_type,
                                    metric_agg=metric_agg,
                                    perm_shuffle=shuffle,
                                )
                                cmd_list.append(cmd_str)
    swarm_path.write_text("\n".join(cmd_list))
    full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

Run sequentially locally

In [ ]:
# import subprocess

# BASE_SAVE_DIR.mkdir(exist_ok=True, parents=True)
# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)
#     break

Run swarm

In [ ]:
run_swarms(
    impact_list=["parent", "self"],
    swarm_log_dir=SWARM_DIR / "logs",
    cmd_dir=SWARM_DIR / "commands",
    n_jobs=N_JOBS,
    n_threads=N_THREADS,
    n_batch=48,
)